# VIscaNEr Stage 2A — frozen visual fusion

Loads the two completed Stage-I DINOv3-B checkpoints, builds a Top-10 + Top-10 candidate union, trains a missing-aware Fusion MLP, and fits CatBoostRanker as a control. Test partitions are read only before training and once after the best `hard_val` checkpoint is selected. OCR columns remain explicitly missing until a real OCR-text dataset is attached.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

INPUT = Path('/kaggle/input')
PROJECT = Path('/kaggle/working/VIscaNEr_fusion')
OUTPUT = Path('/kaggle/working/fusion_stage2a')
OCR_CSV = None  # optional attached CSV with source_relative_path,ocr_text
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLBACKEND'] = 'Agg'

def dataset_root(slug, marker):
    roots = set()
    for match in INPUT.rglob(marker):
        for parent in (match, *match.parents):
            if parent.name == slug:
                roots.add(parent)
                break
    if len(roots) != 1:
        raise RuntimeError(f'Attach exactly one dataset {slug} containing {marker}; found: {sorted(roots)}')
    return roots.pop()

def stage1_checkpoint(kind):
    run_name = f'dinov3_vitb16_{kind}_hard_finetune'
    matches = [p for p in INPUT.rglob('best.pt') if run_name in p.parts]
    if len(matches) != 1:
        raise RuntimeError(f'Attach the completed Stage-I {kind} notebook output; found: {matches}')
    return matches[0]

code = dataset_root('viscaner-fusion-stage2-code', 'code_sha256.json')
hard = dataset_root('viscaner-fusion-hardset-v1', 'manifest.csv')
labels = dataset_root('viscaner-dinov3-data', 'crops_metadata.csv')
bottles = dataset_root('viscaner-bottle-classifier-data', 'audit_metadata.csv')
label_checkpoint = stage1_checkpoint('labels')
bottle_checkpoint = stage1_checkpoint('bottles')
shutil.rmtree(PROJECT, ignore_errors=True)
shutil.copytree(code, PROJECT, ignore=shutil.ignore_patterns('dataset-metadata.json'))
for archive in PROJECT.glob('*.zip'):
    destination = PROJECT / archive.stem
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(destination)
checksums = json.loads((PROJECT/'code_sha256.json').read_text())
for name, expected in checksums.items():
    actual = hashlib.sha256((PROJECT/name).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Code checksum mismatch: {name}')
print(f'SETUP | verified {len(checksums)} versioned code files', flush=True)
sys.path.insert(0, str(PROJECT))
from fusion_stage2.kaggle_io import materialize_dataset_dirs
cache = Path('/kaggle/working/dataset_cache')
hard = materialize_dataset_dirs(hard, ('label_crops', 'bottle_crops'), cache)
labels = materialize_dataset_dirs(labels, ('refs', 'crops'), cache)
bottles = materialize_dataset_dirs(bottles, ('refs', 'crops'), cache)
for required in (hard/'manifest.csv', hard/'label_crops', hard/'bottle_crops', labels/'refs', labels/'crops', bottles/'refs', bottles/'crops', label_checkpoint, bottle_checkpoint):
    if not required.exists():
        raise FileNotFoundError(required)
print('SETUP | all attached datasets and both Stage-I checkpoints resolved', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=5.6,<6', 'safetensors>=0.5,<1', 'catboost>=1.2,<2', 'rapidfuzz>=3,<4', 'Unidecode>=1.3,<2'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a Kaggle GPU accelerator before running this notebook')
print('SETUP | GPU:', torch.cuda.get_device_name(0), '| Stage 2A uses one GPU', flush=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-u', '-m', 'fusion_stage2.train_fusion',
    '--stage', 'frozen',
    '--manifest', str(hard/'manifest.csv'), '--hard-root', str(hard),
    '--label-data-root', str(labels), '--bottle-data-root', str(bottles),
    '--label-refs-root', str(labels/'refs'), '--bottle-refs-root', str(bottles/'refs'),
    '--label-checkpoint', str(label_checkpoint), '--bottle-checkpoint', str(bottle_checkpoint),
    '--output-dir', str(OUTPUT), '--device', 'cuda', '--top-k', '10',
    '--frozen-epochs', '40', '--min-epochs', '6', '--patience', '6',
    '--eval-batch-size', '64', '--fusion-batch-size', '512', '--catboost',
]
if OCR_CSV:
    command += ['--ocr-csv', str(OCR_CSV)]
print('TRAINING | Stage 2A: frozen DINO-B x2 + Fusion MLP; monitor=hard_val Recall@1', flush=True)
subprocess.run(command, cwd=PROJECT, check=True)
summary = json.loads((OUTPUT/'frozen_final_metrics.json').read_text())
print('FINAL STAGE 2A |', json.dumps(summary, ensure_ascii=False), flush=True)
